# PhoBERT + LLM Explanation — Validation Quality

**Branch**: `master`  
**Mục đích**: Cung cấp 3 bằng chứng độc lập "LLM không bịa" cho báo cáo.

| Method | Claim | Metric | N |
|--------|-------|--------|---|
| M1 — Label Consistency | LLM không bịa label | % khớp sentiment với PhoBERT | 200 |
| M2 — BERTScore Evidence | LLM không bịa câu trích | Mean BERTScore F1(evidence, review) | 200 |
| M3 — Human Rubric | LLM không bịa diễn giải | Mean Faithfulness & Usefulness (0–2) | 30 |

**Yêu cầu Kaggle Secrets**:
- `GITHUB_TOKEN` — để clone private repo
- `OPENAI_API_KEY` — để generate LLM explanations

**Yêu cầu Kaggle Dataset**:
- Dataset tên `phobert-absa-hotel` chứa file `best_model.pt`
  (Lấy từ `outputs/results/phobert_best_single/models_cls_only/best_model.pt` trong Group16_report_code.zip)

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

pkgs = [
    'transformers==4.41.0',
    'underthesea',
    'py_vncorenlp',
    'sentencepiece',
    'openai',
    'bert-score',
    'tabulate',
    'tqdm',
    'scikit-learn',
]
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    print('[WARN] Không có GPU — BERTScore sẽ chạy chậm hơn trên CPU')

print('Dependencies installed')

In [ ]:
# Cell 2 — Clone master branch + setup paths
import os, sys

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    REPO_URL = f'https://{token}@github.com/vudinhminh08/NLP-project-master-study.git'
    print('GitHub token loaded from Kaggle Secrets')
except Exception as e:
    print(f'[WARN] Không lấy được token: {e}')
    REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
    print('Fallback: public URL')

REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    print(f'Cloning (branch: {REPO_BRANCH})...')
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print('Clone completed')
else:
    print('Repo đã tồn tại — pulling latest...')
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for d in ['data', 'outputs/results/phobert_best_single/results_cls_only',
          'outputs/results/llm_explainability', 'outputs/llm_cache']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, 'code/data_processing')
sys.path.insert(0, 'code/phobert')
sys.path.insert(0, 'code/llm_rag')
sys.path.insert(0, 'code/llm_explainability')

print('\nCode structure:')
!ls code/llm_explainability/ code/phobert/

In [ ]:
# Cell 3 — API keys + checkpoint path
import os

# ── OpenAI API Key ──
try:
    from kaggle_secrets import UserSecretsClient
    OPENAI_API_KEY = UserSecretsClient().get_secret('OPENAI_API_KEY')
    print('OpenAI API key loaded from Kaggle Secrets')
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    raise ValueError(
        'Thiếu OPENAI_API_KEY.\n'
        '→ Thêm Kaggle Secret: Add-ons → Secrets → OPENAI_API_KEY'
    )
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('OPENAI_API_KEY: OK')

# ── PhoBERT Checkpoint ──
# Option 1: Upload best_model.pt lên Kaggle dataset tên 'phobert-absa-hotel'
# Option 2: Nếu checkpoint có trong repo (uncommonly committed)
CHECKPOINT_PATH = '/kaggle/input/phobert-absa-hotel/best_model.pt'
if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = 'outputs/results/phobert_best_single/models_cls_only/best_model.pt'

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        'Không tìm thấy PhoBERT checkpoint!\n'
        '→ Upload best_model.pt lên Kaggle Dataset tên "phobert-absa-hotel"\n'
        '   File này nằm trong Group16_report_code.zip: '
        'outputs/results/phobert_best_single/models_cls_only/best_model.pt'
    )

print(f'Checkpoint: {CHECKPOINT_PATH}')

In [ ]:
# Cell 4 — Load PhoBERT checkpoint + predict trên test set
# Output: y_pred (numpy array [N_test, 34]) + records cho LLM
import os, sys, json
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from utils.constants import TRAIN_CONFIG, PHOBERT_MODEL_NAME
from utils.helpers import get_device, load_json
from step2_dataloader import create_dataloaders
from model import ABSAPhoBERT
from predict import load_best_model, predict_and_evaluate
from train import load_class_weights
from prediction_formatter import prediction_matrix_to_records

N_EXPLAIN = 200  # số sample dùng cho explanation + validation

device         = get_device()
encoder_option = 'cls_only'
max_seq_len    = TRAIN_CONFIG['max_seq_len']   # 256

print(f'Device: {device} | Encoder: {encoder_option} | SeqLen: {max_seq_len}')

# Tokenizer
print(f'Loading tokenizer: {PHOBERT_MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL_NAME)

# DataLoaders (chỉ cần test_loader)
_, _, test_loader = create_dataloaders(
    train_path = 'data/train_preprocessed.csv',
    dev_path   = 'data/dev_preprocessed.csv',
    test_path  = 'data/test_preprocessed.csv',
    tokenizer  = tokenizer,
    batch_size = 16,
    max_len    = max_seq_len,
    num_workers= 2,
    use_preprocessed = True,
)

# Class weights
class_weights = load_class_weights(
    'outputs/eda/class_weights.json',
    weight_clip = TRAIN_CONFIG['weight_clip'],
    device      = torch.device('cpu'),
)

# Model
print(f'Building ABSAPhoBERT ({encoder_option})...')
model = ABSAPhoBERT(
    model_name     = PHOBERT_MODEL_NAME,
    dropout        = TRAIN_CONFIG['dropout'],
    encoder_option = encoder_option,
).to(device)
model = load_best_model(CHECKPOINT_PATH, model, device)

# Predict
PRED_SAVE = 'outputs/results/phobert_best_single/results_cls_only/phobert_test_metrics.json'
test_metrics, y_true, y_pred = predict_and_evaluate(
    model, test_loader, class_weights, device,
    split_name = 'test',
    save_path  = PRED_SAVE,
)

y_pred = np.array(y_pred)   # shape: [N_test, 34]

print(f'\nTest ACD F1:      {test_metrics["macro_acd_f1"]:.4f}')
print(f'Test SPC F1:      {test_metrics["macro_spc_f1"]:.4f}')
print(f'Test Combined F1: {test_metrics["macro_combined_f1"]:.4f}  (baseline: 0.5543)')
print(f'y_pred shape:     {y_pred.shape}')

# Tạo records cho LLM explanation (dùng Review gốc, không phải processed)
test_df  = pd.read_csv('data/test.csv')
reviews  = test_df['Review'].astype(str).tolist()
records  = prediction_matrix_to_records(
    reviews    = reviews,
    pred_matrix= y_pred,
    max_samples= N_EXPLAIN,
)
print(f'\nRecords for LLM explanation: {len(records)} samples')
print(f'Example predictions (sample 0):')
for p in records[0]["predictions"][:3]:
    print(f'  {p["aspect"]:40s} → {p["sentiment"]}')

In [ ]:
# Cell 5 — Generate LLM Explanations (N=200)
# Dùng GPT-4o-mini | Cache kết quả để tránh gọi lại API
import os, json, time

from llm_client import LLMClient
from llm_explainer import explain_batch

SAMPLES_PATH = 'outputs/results/llm_explainability/explanation_samples.json'
FORCE_REGENERATE = False  # Set True để chạy lại dù đã có cache

if os.path.exists(SAMPLES_PATH) and not FORCE_REGENERATE:
    print(f'Cache found: {SAMPLES_PATH}')
    with open(SAMPLES_PATH, encoding='utf-8') as f:
        data = json.load(f)
    samples = data.get('samples', data)
    print(f'Loaded {len(samples)} cached explanation samples')
else:
    print(f'Generating LLM explanations for {len(records)} samples...')
    print('Estimated time: ~10–20 min (GPT-4o-mini, N=200, 0.5s sleep/call)')

    client = LLMClient(
        provider = 'openai',
        api_key  = os.environ['OPENAI_API_KEY'],
        model    = 'gpt-4o-mini',
        cache_dir= 'outputs/llm_cache',
    )

    outputs, stats = explain_batch(
        records    = records,
        llm_client = client,
        max_samples= N_EXPLAIN,
        sleep_sec  = 0.5,
    )

    os.makedirs(os.path.dirname(SAMPLES_PATH), exist_ok=True)
    with open(SAMPLES_PATH, 'w', encoding='utf-8') as f:
        json.dump({'samples': outputs}, f, ensure_ascii=False, indent=2)

    STATS_PATH = 'outputs/results/llm_explainability/explanation_quality_report.json'
    with open(STATS_PATH, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    samples = outputs
    print(f'\n=== Generation Stats ===')
    print(json.dumps(stats, ensure_ascii=False, indent=2))

print(f'\nTotal samples: {len(samples)}')
# Quick sanity check
s0 = samples[0]
print(f'Sample 0 review: {s0["review"][:80]}...')
print(f'Items count: {len(s0["explanation"]["items"])}')
if s0['explanation']['items']:
    item0 = s0['explanation']['items'][0]
    print(f'Item 0: {item0["aspect"]} | {item0["sentiment"]} | evidence: "{item0.get("evidence","")[:60]}"')

In [ ]:
# Cell 6 — Method 1: Label Consistency
# Claim: LLM không bịa label (aspect + sentiment)
from validate_llm_quality import compute_label_consistency

print('=' * 60)
print('Method 1 — Label Consistency (N=200)')
print('=' * 60)

r1 = compute_label_consistency(samples, n=200)

print(f'  Samples processed      : {r1["n_samples"]} (parse_fail skipped: {r1["n_parse_fail"]})')
print(f'  Total LLM items        : {r1["total_llm_items_generated"]}')
print(f'  Dropped spurious aspect: {r1["total_dropped_spurious"]} items')
print(f'  Sentiment mismatch     : {r1["total_sentiment_wrong"]} items')
print()
print(f'  Aspect Preservation Rate   : {r1["aspect_preservation_rate"]}%')
print(f'  Sentiment Consistency Rate : {r1["sentiment_consistency_rate"]}%')
print(f'  Overall Label Accuracy     : {r1["overall_label_accuracy"]}%')
print()
print('→ Kết luận: Label Consistency cao → LLM không bịa aspect/sentiment.')

In [ ]:
# Cell 7 — Method 2: BERTScore Evidence Groundedness
# Claim: LLM không bịa câu trích (evidence)
# Backbone: vinai/phobert-base-v2 | Citation: Zhang et al. 2020
# Note: cell này chạy ~5–10 phút (download PhoBERT weights ~500MB, rồi compute)
from validate_llm_quality import compute_bertscore_groundedness

print('=' * 60)
print('Method 2 — BERTScore Evidence Groundedness (N=200)')
print('=' * 60)
print('Note: Download PhoBERT weights lần đầu (~500MB), sau đó cached.')

r2 = compute_bertscore_groundedness(
    samples    = samples,
    n          = 200,
    model_type = 'vinai/phobert-base-v2',
    batch_size = 64,
    verbose    = True,
)

print(f'\n  Evidence items scored : {r2["n_evidence_items"]}')
print(f'  Backbone              : {r2["model_type"]}')
print()
print(f'  Mean BERTScore Precision : {r2["mean_bertscore_precision"]:.4f}')
print(f'  Mean BERTScore Recall    : {r2["mean_bertscore_recall"]:.4f}')
print(f'  Mean BERTScore F1        : {r2["mean_bertscore_f1"]:.4f}  ← số liệu báo cáo')
print()
print(f'  Citation: {r2["citation"]}')
print()
print('→ Kết luận: F1 cao → Evidence LLM trích thật sự từ review, không bịa.')

In [ ]:
# Cell 8 — Method 3: Human Rubric Template (N=30)
# Claim: LLM không bịa nội dung diễn giải
# Step 1: Tạo template CSV → annotation thủ công offline
# Step 2: Sau khi điền, chạy lại cell này với annotated CSV
import os
from validate_llm_quality import generate_rubric_template, compute_rubric_scores

RUBRIC_TEMPLATE  = 'outputs/results/llm_explainability/rubric_template.csv'
RUBRIC_ANNOTATED = 'outputs/results/llm_explainability/rubric_annotated.csv'

print('=' * 60)
print('Method 3 — Human Rubric (N=30)')
print('=' * 60)

# Tạo template
generate_rubric_template(samples, n=30, save_path=RUBRIC_TEMPLATE, seed=42)

# Nếu đã có annotation → tính kết quả luôn
if os.path.exists(RUBRIC_ANNOTATED):
    print('\n→ Tìm thấy annotated file — tính kết quả:')
    r3 = compute_rubric_scores(RUBRIC_ANNOTATED)
    print(f'  N annotated      : {r3["n_annotated"]}')
    print(f'  Mean Faithfulness: {r3["mean_faithfulness"]} ± {r3["std_faithfulness"]} / 2.0  ({r3["pct_fully_faithful"]}% score 2)')
    print(f'  Mean Usefulness  : {r3["mean_usefulness"]} ± {r3["std_usefulness"]} / 2.0  ({r3["pct_fully_useful"]}% score 2)')
    print(f'  Faithfulness dist: {r3["faithfulness_dist"]}')
    print(f'  Usefulness dist  : {r3["usefulness_dist"]}')
else:
    print('\n[NEXT STEP] Điền annotation vào rubric_template.csv:')
    print(f'  File: {RUBRIC_TEMPLATE}')
    print('  Columns để điền: faithfulness (0/1/2), usefulness (0/1/2)')
    print(f'  Sau khi điền xong, lưu thành: {RUBRIC_ANNOTATED}')
    print('  Rồi chạy lại cell này để ra kết quả.')

In [ ]:
# Cell 9 — Summary + Save Validation Report
import json, os
import pandas as pd
from validate_llm_quality import compute_rubric_scores

RUBRIC_ANNOTATED = 'outputs/results/llm_explainability/rubric_annotated.csv'

validation_results = {
    'label_consistency':      r1,
    'bertscore_groundedness': r2,
}
if os.path.exists(RUBRIC_ANNOTATED):
    validation_results['human_rubric'] = compute_rubric_scores(RUBRIC_ANNOTATED)

# Save
REPORT_PATH = 'outputs/results/llm_explainability/validation_report.json'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(validation_results, f, ensure_ascii=False, indent=2)
print(f'Validation report saved: {REPORT_PATH}')

# Print summary table
print('\n' + '=' * 65)
print('VALIDATION SUMMARY — LLM Non-Hallucination Evidence')
print('=' * 65)

rows = [
    ['M1 — Label Consistency (N=200)',
     'Sentiment consistency',
     f"{r1['sentiment_consistency_rate']}%",
     'LLM không bịa label'],
    ['M2 — BERTScore Evidence (N=200)',
     'Mean BERTScore F1',
     f"{r2['mean_bertscore_f1']:.4f}",
     'LLM không bịa câu trích'],
]

if 'human_rubric' in validation_results:
    r3 = validation_results['human_rubric']
    rows.append([
        'M3 — Human Rubric (N=30)',
        'Faithfulness / Usefulness',
        f"{r3['mean_faithfulness']:.2f} / {r3['mean_usefulness']:.2f} (out of 2)",
        'LLM không bịa diễn giải'
    ])
else:
    rows.append([
        'M3 — Human Rubric (N=30)',
        'Faithfulness / Usefulness',
        '(chờ annotation)',
        'LLM không bịa diễn giải'
    ])

df_summary = pd.DataFrame(rows, columns=['Method', 'Metric', 'Value', 'Claim'])
print(df_summary.to_string(index=False))
print('=' * 65)

print('\n→ 3 bằng chứng độc lập cho 3 loại bịa khác nhau:')
print('   M1 = không bịa label    → dùng metadata từ validate_explanation_items()')
print('   M2 = không bịa evidence → BERTScore F1(evidence, review) cao')
print('   M3 = không bịa diễn giải → human judge Faithfulness/Usefulness')

# Zip để download
import shutil
shutil.make_archive('/kaggle/working/llm_validation_results', 'zip',
                    'outputs/results/llm_explainability')
print('\nDownload: /kaggle/working/llm_validation_results.zip → Kaggle Output')